# Multi-Frame CRNN Training Walkthrough

This notebook breaks down the training pipeline for a Multi-Frame CRNN model used for text recognition from video tracks. We will go through each component step-by-step.

## 1. Imports and Setup
First, we import necessary libraries and set a random seed for reproducibility.

In [3]:
import os
import glob
import json
import random
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image
import numpy as np
from tqdm import tqdm
from torch.amp import autocast, GradScaler
import albumentations as A
from albumentations.pytorch import ToTensorV2
import cv2
import torchvision.models as models

def seed_everything(seed=42):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    print(f"🔒 Đã cố định Seed: {seed}")

## 2. Configuration
Define all hyperparameters and constants in a Config class.

In [ ]:
class Config:
    DATA_ROOT = "/Users/nguyenthientai/Documents/baseline_icpr_2026-main/train" # Nhớ sửa lại đường dẫn đúng trên máy bạn
    IMG_HEIGHT = 32
    IMG_WIDTH = 128
    CHARS = "0123456789ABCDEFGHIJKLMNOPQRSTUVWXYZ-"
    BATCH_SIZE = 64
    
    # ⚠️ LƯU Ý KHI TRAIN TIẾP:
    # Nếu train tiếp, bạn nên giảm Learning Rate xuống một chút (ví dụ: từ 0.001 -> 0.0005 hoặc 0.0001)
    # để tránh làm hỏng các trọng số đã học tốt từ trước (catastrophic forgetting).
    LEARNING_RATE = 0.001 
    EPOCHS = 50
    SEED = 42
    
    # --- CẤU HÌNH RESUME ---
    # RESUME_TRAINING = True  # Set True để load weight cũ
    # WEIGHTS_PATH = "/Users/nguyenthientai/Documents/baseline_icpr_2026-main/best_model1.pth" # Đường dẫn đến file weight bạn muốn dùng
    # -----------------------

    # Cấu hình thiết bị (giữ nguyên như trước)
    if torch.backends.mps.is_available():
        DEVICE = torch.device('mps')
        print("🍏 Đã kích hoạt MPS (Metal Performance Shaders) cho Macbook!")
    elif torch.cuda.is_available():
        DEVICE = torch.device('cuda')
        print("🔥 Đã kích hoạt CUDA!")
    else:
        DEVICE = torch.device('cpu')
        print("⚠️ Không tìm thấy GPU, đang chạy trên CPU!")

    CHAR2IDX = {char: idx + 1 for idx, char in enumerate(CHARS)}
    IDX2CHAR = {idx + 1: char for idx, char in enumerate(CHARS)}
    NUM_CLASSES = len(CHARS) + 1
    VAL_SPLIT_FILE = "val_tracks.json"
    TEST_SPLIT_FILE = "test_tracks.json"
    VAL_SIZE = 2000
    TEST_SIZE = 2000

🍏 Đã kích hoạt MPS (Metal Performance Shaders) cho Macbook!


: 

: 

: 

## 3. Data Augmentation
Define transformations for training (with augmentation) and validation/testing.

In [ ]:
def get_train_transforms():
    return A.Compose([
        A.Resize(height=Config.IMG_HEIGHT, width=Config.IMG_WIDTH),

        # FIX: cval -> fill
        A.Affine(scale=(0.95, 1.05), translate_percent=(0.05, 0.05), rotate=(-5, 5), p=0.5, fill=128),
        A.Perspective(scale=(0.02, 0.05), p=0.3),

        A.RandomBrightnessContrast(p=0.5),
        A.HueSaturationValue(hue_shift_limit=10, sat_shift_limit=20, val_shift_limit=20, p=0.3),

        A.CoarseDropout(num_holes_range=(1, 3), hole_height_range=(4, 8), hole_width_range=(4, 8), p=0.3),

        A.Normalize(mean=(0.5, 0.5, 0.5), std=(0.5, 0.5, 0.5)),
        ToTensorV2()
    ])

def get_degradation_transforms():
    return A.Compose([
        A.OneOf([
            A.GaussianBlur(blur_limit=(3, 7), p=1.0),
            A.MotionBlur(blur_limit=(3, 7), p=1.0),
            A.Defocus(radius=(1, 3), alias_blur=(0.1, 0.3), p=1.0),
        ], p=0.8),

        A.OneOf([
            A.GaussNoise(var_limit=(10.0, 50.0), p=1.0),
            A.ISONoise(color_shift=(0.01, 0.05), intensity=(0.1, 0.5), p=1.0),
            A.MultiplicativeNoise(multiplier=(0.9, 1.1), p=1.0),
        ], p=0.8),

        A.ImageCompression(quality_range=(10, 50), p=0.5),
        A.Downscale(scale_range=(0.25, 0.5), p=0.5),
    ])

def get_val_transforms():
    return A.Compose([
        A.Resize(height=Config.IMG_HEIGHT, width=Config.IMG_WIDTH),
        A.Normalize(mean=(0.5, 0.5, 0.5), std=(0.5, 0.5, 0.5)),
        ToTensorV2()
    ])

: 

: 

: 

## 4. Dataset Class
Custom Dataset class to handle multi-frame loading and splitting.

In [ ]:
class AdvancedMultiFrameDataset(Dataset):
    def __init__(self, root_dir, mode='train'):
        self.mode = mode
        self.samples = []

        if mode == 'train':
            self.transform = get_train_transforms()
            self.degrade = get_degradation_transforms()
        else:
            self.transform = get_val_transforms()
            self.degrade = None

        print(f"[{mode.upper()}] Scanning: {root_dir}")
        abs_root = os.path.abspath(root_dir)
        search_path = os.path.join(abs_root, "**", "track_*")
        all_tracks = sorted(glob.glob(search_path, recursive=True))

        if not all_tracks:
            print("❌ LỖI: Không tìm thấy data.")
            return

        train_tracks = []
        val_tracks = []
        test_tracks = []

        # Check if split files exist
        val_exists = os.path.exists(Config.VAL_SPLIT_FILE)
        test_exists = os.path.exists(Config.TEST_SPLIT_FILE)

        val_ids = set()
        test_ids = set()

        if val_exists and test_exists:
            print(f"📂 Loading splits from '{Config.VAL_SPLIT_FILE}' and '{Config.TEST_SPLIT_FILE}'...")
            try:
                with open(Config.VAL_SPLIT_FILE, 'r') as f:
                    val_ids = set(json.load(f))
                with open(Config.TEST_SPLIT_FILE, 'r') as f:
                    test_ids = set(json.load(f))
            except:
                val_ids = set()
                test_ids = set()
                print("⚠️ Lỗi đọc file split, sẽ tạo lại.")

            for t in all_tracks:
                track_name = os.path.basename(t)
                if track_name in val_ids:
                    val_tracks.append(t)
                elif track_name in test_ids:
                    test_tracks.append(t)
                else:
                    train_tracks.append(t)

            # Nếu split không khớp, tạo lại
            if (not val_tracks or not test_tracks) and len(all_tracks) > 0:
                print("⚠️ File split không khớp với dữ liệu hiện tại. Chia lại...")
                val_ids = set()
                test_ids = set()

        if not val_ids or not test_ids:
            print(f"⚠️ Creating new split: {Config.VAL_SIZE} val, {Config.TEST_SIZE} test...")
            random.Random(Config.SEED).shuffle(all_tracks)

            # Chia: val 2000, test 2000, train còn lại
            val_tracks = all_tracks[:Config.VAL_SIZE]
            test_tracks = all_tracks[Config.VAL_SIZE:Config.VAL_SIZE + Config.TEST_SIZE]
            train_tracks = all_tracks[Config.VAL_SIZE + Config.TEST_SIZE:]

            # Lưu split files
            val_ids = [os.path.basename(t) for t in val_tracks]
            test_ids = [os.path.basename(t) for t in test_tracks]
            with open(Config.VAL_SPLIT_FILE, 'w') as f:
                json.dump(val_ids, f, indent=2)
            with open(Config.TEST_SPLIT_FILE, 'w') as f:
                json.dump(test_ids, f, indent=2)
            print(f"✅ Saved splits: val={len(val_tracks)}, test={len(test_tracks)}, train={len(train_tracks)}")

        if mode == 'train':
            selected_tracks = train_tracks
        elif mode == 'val':
            selected_tracks = val_tracks
        else:  # mode == 'test'
            selected_tracks = test_tracks
        print(f"[{mode.upper()}] Loaded {len(selected_tracks)} tracks.")

        for track_path in tqdm(selected_tracks, desc=f"Indexing {mode}"):
            json_path = os.path.join(track_path, "annotations.json")
            if not os.path.exists(json_path): continue
            try:
                with open(json_path, 'r') as f:
                    data = json.load(f)
                if isinstance(data, list): data = data[0]
                label = data.get('plate_text', data.get('license_plate', data.get('text', '')))
                if not label: continue

                lr_files = sorted(glob.glob(os.path.join(track_path, "lr-*.png")) + glob.glob(os.path.join(track_path, "lr-*.jpg")))
                hr_files = sorted(glob.glob(os.path.join(track_path, "hr-*.png")) + glob.glob(os.path.join(track_path, "hr-*.jpg")))

                if len(lr_files) > 0:
                    self.samples.append({
                        'lr_paths': lr_files,
                        'hr_paths': hr_files,
                        'label': label
                    })
            except: pass

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        item = self.samples[idx]
        label = item['label']

        use_hr = (self.mode == 'train') and (len(item['hr_paths']) > 0) and (random.random() < 0.5)

        if use_hr:
            img_paths = item['hr_paths']
            if len(img_paths) < 5: img_paths = img_paths + [img_paths[-1]] * (5 - len(img_paths))
            else: img_paths = img_paths[:5]

            images_list = []
            for p in img_paths:
                image = cv2.imread(p)
                if image is None: image = np.zeros((Config.IMG_HEIGHT, Config.IMG_WIDTH, 3), dtype=np.uint8)
                else: image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

                if self.degrade:
                    image = self.degrade(image=image)['image']
                image = self.transform(image=image)['image']
                images_list.append(image)
        else:
            img_paths = item['lr_paths']
            if len(img_paths) < 5: img_paths = img_paths + [img_paths[-1]] * (5 - len(img_paths))
            else: img_paths = img_paths[:5]

            images_list = []
            for p in img_paths:
                image = cv2.imread(p)
                if image is None: image = np.zeros((Config.IMG_HEIGHT, Config.IMG_WIDTH, 3), dtype=np.uint8)
                else: image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

                image = self.transform(image=image)['image']
                images_list.append(image)

        images_tensor = torch.stack(images_list, dim=0)
        target = [Config.CHAR2IDX[c] for c in label if c in Config.CHAR2IDX]
        if len(target) == 0: target = [0]

        return images_tensor, torch.tensor(target, dtype=torch.long), len(target), label

    @staticmethod
    def collate_fn(batch):
        images, targets, target_lengths, labels_text = zip(*batch)
        images = torch.stack(images, 0)
        targets = torch.cat(targets)
        target_lengths = torch.tensor(target_lengths, dtype=torch.long)
        return images, targets, target_lengths, labels_text

: 

: 

: 

## 5. Model Architecture
Define the Attention Fusion module and the main CRNN model.

In [ ]:
class AttentionFusion(nn.Module):
    def __init__(self, channels):
        super().__init__()
        self.score_net = nn.Sequential(
            nn.Conv2d(channels, channels // 8, kernel_size=1),
            nn.ReLU(True),
            nn.Conv2d(channels // 8, 1, kernel_size=1)
        )
    def forward(self, x):
        b_frames, c, h, w = x.size()
        b_size = b_frames // 5
        x_view = x.view(b_size, 5, c, w)
        scores = self.score_net(x).view(b_size, 5, 1, w)
        return torch.sum(x_view * F.softmax(scores, dim=1), dim=1)

class MultiFrameCRNN(nn.Module):
    def __init__(self, num_classes, hidden_size=256):
        super().__init__()
        # self.cnn = nn.Sequential(
        #     nn.Conv2d(3, 64, 3, 1, 1), nn.ReLU(True), nn.MaxPool2d(2, 2),
        #     nn.Conv2d(64, 128, 3, 1, 1), nn.ReLU(True), nn.MaxPool2d(2, 2),
        #     nn.Conv2d(128, 256, 3, 1, 1), nn.BatchNorm2d(256), nn.ReLU(True),
        #     nn.Conv2d(256, 256, 3, 1, 1), nn.ReLU(True), nn.MaxPool2d((2, 2), (2, 1), (0, 1)),
        #     nn.Conv2d(256, 512, 3, 1, 1), nn.BatchNorm2d(512), nn.ReLU(True),
        #     nn.Conv2d(512, 512, 3, 1, 1), nn.ReLU(True), nn.MaxPool2d((2, 2), (2, 1), (0, 1)),
        #     nn.Conv2d(512, 512, 2, 1, 0), nn.BatchNorm2d(512), nn.ReLU(True)
        # )
        
        effnet = models.efficientnet_b0(weights=models.EfficientNet_B0_Weights.DEFAULT)
        self.backbone = effnet.features
        self.adaption = nn.Sequential(
            nn.Conv2d(1280, 512, kernel_size=1, bias=False),
            nn.BatchNorm2d(512),
            nn.ReLU(inplace=True)
        )
        
        self.fusion = AttentionFusion(channels=512)
        self.rnn = nn.Sequential(nn.LSTM(512, hidden_size, bidirectional=True, batch_first=True, num_layers=2, dropout=0.25))
        self.fc = nn.Linear(hidden_size * 2, num_classes)

    def forward(self, x):
        b, t, c, h, w = x.size()
        x = x.view(b * t, c, h, w)
        feat = self.backbone(x)
        feat = self.adaption(feat)
        fused = self.fusion(feat)
        fused = F.interpolate(fused, size=(1, 32), mode='bilinear', align_corners=False)
        rnn_in = fused.permute(0, 3, 1, 2).squeeze(-1)
        # out = self.fc(self.rnn(fused.permute(0, 2, 1))[0])
        out, _ = self.rnn(rnn_in)
        out = self.fc(out)
        return out.log_softmax(2)

: 

: 

: 

## 6. Utility Functions
Helper function to decode model predictions into text.

In [ ]:
def decode_predictions(preds, idx2char):
    result_list = []
    for p in preds:
        pred_str = ""
        last_char = 0
        for char_idx in p:
            c = char_idx.item()
            if c != 0 and c != last_char: pred_str += idx2char[c]
            last_char = c
        result_list.append(pred_str)
    return result_list

: 

: 

: 

## 7. Training Pipeline
The main training loop, including validation and testing.

In [ ]:
def train_pipeline():
    seed_everything(Config.SEED)
    print(f"🚀 TRAINING START | Device: {Config.DEVICE}")

    if not os.path.exists(Config.DATA_ROOT):
        print("❌ LỖI: Sai đường dẫn DATA_ROOT. Hãy kiểm tra lại Config.DATA_ROOT")
        return

    train_ds = AdvancedMultiFrameDataset(Config.DATA_ROOT, mode='train')
    val_ds = AdvancedMultiFrameDataset(Config.DATA_ROOT, mode='val')
    test_ds = AdvancedMultiFrameDataset(Config.DATA_ROOT, mode='test')

    if len(train_ds) == 0:
        print("❌ Dataset Train rỗng!")
        return

    # MPS: num_workers=2 hoặc 0 nếu lỗi
    train_loader = DataLoader(train_ds, batch_size=Config.BATCH_SIZE, shuffle=True,
                               collate_fn=AdvancedMultiFrameDataset.collate_fn, num_workers=0, pin_memory=True)

    val_loader = DataLoader(val_ds, batch_size=Config.BATCH_SIZE, shuffle=False,
                            collate_fn=AdvancedMultiFrameDataset.collate_fn, num_workers=0, pin_memory=True) if len(val_ds) > 0 else None
    
    test_loader = DataLoader(test_ds, batch_size=Config.BATCH_SIZE, shuffle=False,
                             collate_fn=AdvancedMultiFrameDataset.collate_fn, num_workers=0, pin_memory=True) if len(test_ds) > 0 else None

    # 1. Khởi tạo model
    model = MultiFrameCRNN(num_classes=Config.NUM_CLASSES).to(Config.DEVICE)
    
    # --- ĐOẠN CODE MỚI: LOAD WEIGHT CŨ ---
    # if Config.RESUME_TRAINING:
    #     if os.path.exists(Config.WEIGHTS_PATH):
    #         print(f"🔄 Đang load weights từ: {Config.WEIGHTS_PATH}")
    #         try:
    #             # Load weights vào đúng thiết bị
    #             checkpoint = torch.load(Config.WEIGHTS_PATH, map_location=Config.DEVICE)
    #             model.load_state_dict(checkpoint)
    #             print("✅ Load weights thành công! Tiếp tục train...")
    #         except Exception as e:
    #             print(f"⚠️ Không thể load weights: {e}")
    #             print("➡️ Sẽ train từ đầu (scratch).")
    #     else:
    #         print(f"⚠️ Không tìm thấy file {Config.WEIGHTS_PATH}. Sẽ train từ đầu.")
    # else:
    #     print("🆕 Training from scratch (từ đầu).")
    # -------------------------------------
    
    # CTCLoss chạy trên CPU
    criterion = nn.CTCLoss(blank=0, zero_infinity=True).to('cpu')
    
    optimizer = optim.AdamW(model.parameters(), lr=Config.LEARNING_RATE, weight_decay=1e-4)
    scheduler = optim.lr_scheduler.OneCycleLR(
        optimizer, 
        max_lr=Config.LEARNING_RATE,
        steps_per_epoch=len(train_loader), 
        epochs=Config.EPOCHS,
        pct_start=0.3,
        div_factor=25.0,
        final_div_factor=1e4)

    # Reset best_acc nếu muốn lưu file mới đè lên, hoặc giữ logic cũ
    # Ở đây ta set về 0 để model phải thực sự tốt hơn mới save đè.
    # Tuy nhiên, nếu model đã rất tốt (ví dụ 90%), epoch đầu tiên của lần train mới (đang warm-up)
    # có thể thấp hơn 90%, nên ta cứ để 0.0 để nó tự tìm best trong session này.
    best_acc = 0.0 
    
    for epoch in range(Config.EPOCHS):
        model.train()
        epoch_loss = 0

        pbar = tqdm(train_loader, desc=f"Ep {epoch+1}/{Config.EPOCHS}")
        for images, targets, target_lengths, _ in pbar:
            images = images.to(Config.DEVICE)
            targets = targets.to('cpu')
            target_lengths = target_lengths.to('cpu')

            optimizer.zero_grad(set_to_none=True)

            preds = model(images) 
            
            # Logic tính toán Loss trên CPU cho MPS
            preds_cpu = preds.to('cpu').float() 
            preds_permuted = preds_cpu.permute(1, 0, 2)
            input_lengths = torch.full(size=(images.size(0),), fill_value=preds.size(1), dtype=torch.long).to('cpu')
            
            loss = criterion(preds_permuted, targets, input_lengths, target_lengths)

            loss.backward()
            optimizer.step()
            scheduler.step()

            epoch_loss += loss.item()
            pbar.set_postfix({'loss': loss.item(), 'lr': scheduler.get_last_lr()[0]})

        avg_train_loss = epoch_loss / len(train_loader)

        # --- VALIDATION ---
        val_acc = 0
        avg_val_loss = 0

        if val_loader:
            model.eval()
            val_loss = 0
            total_correct = 0
            total_samples = 0

            with torch.no_grad():
                for images, targets, target_lengths, labels_text in val_loader:
                    images = images.to(Config.DEVICE)
                    targets = targets.to('cpu')
                    target_lengths = target_lengths.to('cpu')
                    
                    preds = model(images)
                    
                    preds_cpu = preds.to('cpu').float()
                    preds_permuted = preds_cpu.permute(1, 0, 2)
                    input_lengths = torch.full((images.size(0),), preds.size(1), dtype=torch.long).to('cpu')
                    
                    loss = criterion(preds_permuted, targets, input_lengths, target_lengths)
                    val_loss += loss.item()

                    decoded = decode_predictions(torch.argmax(preds_cpu, dim=2), Config.IDX2CHAR)
                    for i in range(len(labels_text)):
                        if decoded[i] == labels_text[i]:
                            total_correct += 1
                    total_samples += len(labels_text)

            avg_val_loss = val_loss / len(val_loader)
            val_acc = (total_correct / total_samples) * 100 if total_samples > 0 else 0

        print(f"Result: Train Loss: {avg_train_loss:.4f} | Val Loss: {avg_val_loss:.4f} | Val Acc: {val_acc:.2f}%")

        if val_acc > best_acc:
            best_acc = val_acc
            # Lưu lại model mới (bạn có thể đổi tên thành best_model_v2.pth nếu sợ mất file cũ)
            torch.save(model.state_dict(), "best_model.pth") 
            print(f" -> ⭐ Saved Best Model! ({val_acc:.2f}%)")

    # --- TESTING ---
    print("\n" + "="*60)
    print("🧪 EVALUATING ON TEST SET...")
    print("="*60)

    if test_loader and os.path.exists("best_model.pth"):
        # Load best model
        model.load_state_dict(torch.load("best_model.pth", weights_only=True))
        model.eval()

        test_correct = 0
        test_total = 0
        test_char_correct = 0
        test_char_total = 0

        results = []  # Lưu kết quả để phân tích

        with torch.no_grad():
            for images, targets, target_lengths, labels_text in tqdm(test_loader, desc="Testing"):
                images = images.to(Config.DEVICE)
                preds = model(images)
                decoded = decode_predictions(torch.argmax(preds, dim=2), Config.IDX2CHAR)

                for i in range(len(labels_text)):
                    gt = labels_text[i]
                    pred = decoded[i]

                    # Exact match accuracy
                    if pred == gt:
                        test_correct += 1
                    test_total += 1

                    # Character-level accuracy
                    for j in range(max(len(gt), len(pred))):
                        test_char_total += 1
                        if j < len(gt) and j < len(pred) and gt[j] == pred[j]:
                            test_char_correct += 1

                    # Lưu một số kết quả sai để debug
                    if pred != gt and len(results) < 20:
                        results.append({'gt': gt, 'pred': pred})

        test_acc = (test_correct / test_total) * 100 if test_total > 0 else 0
        char_acc = (test_char_correct / test_char_total) * 100 if test_char_total > 0 else 0

        print(f"\n📊 TEST RESULTS:")
        print(f"   • Exact Match Accuracy: {test_acc:.2f}% ({test_correct}/{test_total})")
        print(f"   • Character Accuracy:   {char_acc:.2f}%")
        print(f"   • Best Val Accuracy:    {best_acc:.2f}%")

        if results:
            print(f"\n🔍 Sample Errors (first 10):")
            for i, r in enumerate(results[:10]):
                print(f"   {i+1}. GT: '{r['gt']}' | Pred: '{r['pred']}'")

        # Lưu kết quả vào file
        test_results = {
            'test_accuracy': test_acc,
            'char_accuracy': char_acc,
            'best_val_accuracy': best_acc,
            'total_samples': test_total,
            'correct_samples': test_correct,
            'sample_errors': results
        }
        with open('test_results.json', 'w') as f:
            json.dump(test_results, f, indent=2)
        print(f"\n💾 Results saved to 'test_results.json'")

    else:
        if not test_loader:
            print("❌ Test loader không có dữ liệu!")
        if not os.path.exists("best_model.pth"):
            print("❌ Không tìm thấy best_model.pth!")

: 

: 

: 

## 8. Run Training
Execute the training pipeline.

In [ ]:
if __name__ == "__main__":
    train_pipeline()

🔒 Đã cố định Seed: 42
🚀 TRAINING START | Device: mps
[TRAIN] Scanning: /Users/nguyenthientai/Documents/baseline_icpr_2026-main/train


/var/folders/8_/bj2z7dxj01x1mz53sdk2mtc40000gn/T/ipykernel_37606/1410680567.py:27: UserWarning: Argument(s) 'var_limit' are not valid for transform GaussNoise
  A.GaussNoise(var_limit=(10.0, 50.0), p=1.0),


📂 Loading splits from 'val_tracks.json' and 'test_tracks.json'...
[TRAIN] Loaded 16000 tracks.


Indexing train: 100%|██████████| 16000/16000 [00:08<00:00, 1973.47it/s]


[VAL] Scanning: /Users/nguyenthientai/Documents/baseline_icpr_2026-main/train
📂 Loading splits from 'val_tracks.json' and 'test_tracks.json'...
[VAL] Loaded 2000 tracks.


...

Ep 1/50:   0%|          | 0/250 [00:00<?, ?it/s]/opt/anaconda3/envs/lrlpr/lib/python3.10/site-packages/torch/utils/data/dataloader.py:692: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  warnings.warn(warn_msg)
Ep 1/50:   0%|          | 0/250 [00:03<?, ?it/s]


RuntimeError: number of dims don't match in permute

: 

: 

: 

In [ ]:
def infer_custom_test_set(test_root_dir, output_file="inference_results.txt", model_path="/Users/nguyenthientai/Documents/baseline_icpr_2026-main/notebook/best_model_tuned2.pth"):
    """
    Infer on a custom test set and save results.
    
    Args:
        test_root_dir: Path to directory containing track_* folders
        output_file: Output file path for results
        model_path: Path to saved model weights
    """
    if not os.path.exists(model_path):
        print(f"❌ Model not found at {model_path}")
        return
    
    # Load model
    model = MultiFrameCRNN(num_classes=Config.NUM_CLASSES).to(Config.DEVICE)
    model.load_state_dict(torch.load(model_path, weights_only=True))
    model.eval()
    
    # Create dataset without splitting
    print(f"📂 Scanning custom test set: {test_root_dir}")
    abs_root = os.path.abspath(test_root_dir)
    search_path = os.path.join(abs_root, "**", "track_*")
    all_tracks = sorted(glob.glob(search_path, recursive=True))
    
    if not all_tracks:
        print(f"❌ No tracks found in {test_root_dir}")
        return
    
    print(f"✅ Found {len(all_tracks)} tracks")
    
    results = []
    
    # Get val transforms (no augmentation)
    transform = get_val_transforms()
    
    with torch.no_grad():
        for track_path in tqdm(all_tracks, desc="Inferencing"):
            track_name = os.path.basename(track_path)
            
            # Load images
            lr_files = sorted(glob.glob(os.path.join(track_path, "lr-*.png")) + 
                            glob.glob(os.path.join(track_path, "lr-*.jpg")))
            hr_files = sorted(glob.glob(os.path.join(track_path, "hr-*.png")) + 
                            glob.glob(os.path.join(track_path, "hr-*.jpg")))
            
            # Prefer HR if available
            img_paths = hr_files if len(hr_files) > 0 else lr_files
            
            if len(img_paths) == 0:
                results.append(f"{track_name},;0.0")
                continue
            
            # Pad to 5 frames
            if len(img_paths) < 5:
                img_paths = img_paths + [img_paths[-1]] * (5 - len(img_paths))
            else:
                img_paths = img_paths[:5]
            
            # Load and process images
            images_list = []
            for p in img_paths:
                image = cv2.imread(p)
                if image is None:
                    image = np.zeros((Config.IMG_HEIGHT, Config.IMG_WIDTH, 3), dtype=np.uint8)
                else:
                    image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
                
                image = transform(image=image)['image']
                images_list.append(image)
            
            images_tensor = torch.stack(images_list, dim=0).unsqueeze(0).to(Config.DEVICE)
            
            # Inference
            preds = model(images_tensor)
            
            # Decode prediction
            pred_logits = torch.argmax(preds, dim=2)[0]  # [seq_len]
            pred_str = ""
            last_char = 0
            confidence_scores = []
            
            # Get confidence from softmax
            preds_probs = F.softmax(preds[0], dim=1)
            
            for i, char_idx in enumerate(pred_logits):
                c = char_idx.item()
                conf = preds_probs[i, c].item()
                confidence_scores.append(conf)
                
                if c != 0 and c != last_char:
                    pred_str += Config.IDX2CHAR.get(c, '')
                last_char = c
            
            # Average confidence
            avg_confidence = np.mean(confidence_scores) if confidence_scores else 0.0
            
            # Format: track_id,plate_text;confidence
            result_line = f"{track_name},{pred_str};{avg_confidence:.4f}"
            results.append(result_line)
    
    # Save results
    with open(output_file, 'w') as f:
        for line in results:
            f.write(line + '\n')
    
    print(f"\n✅ Inference complete! Results saved to '{output_file}'")
    print(f"📊 Total predictions: {len(results)}")
    
    # Print sample results
    print(f"\n📋 Sample Results (first 10):")
    for line in results[:10]:
        print(f"   {line}")
    
    return results

: 

: 

: 

In [ ]:
# ==========================================
# CONFIGURE THESE PARAMETERS
# ==========================================
CUSTOM_TEST_DIR = "/Users/nguyenthientai/Documents/baseline_icpr_2026-main/pub_test"
OUTPUT_FILE = "inference_results2.txt"       # Output file name

# ==========================================
# Run Inference
# ==========================================
if os.path.exists(CUSTOM_TEST_DIR):
    print("\n" + "="*60)
    print("🔍 RUNNING INFERENCE ON CUSTOM TEST SET")
    print("="*60)
    results = infer_custom_test_set(CUSTOM_TEST_DIR, output_file=OUTPUT_FILE)
    
    # Print summary statistics
    print("\n" + "="*60)
    print("📊 INFERENCE RESULTS SUMMARY")
    print("="*60)
    
    total_results = len(results)
    plates_detected = sum(1 for r in results if r.split(',')[1].split(';')[0])  # Non-empty plate
    avg_confidence = np.mean([float(r.split(';')[1]) for r in results if ';' in r])
    
    print(f"✅ Total Tracks: {total_results}")
    print(f"✅ Plates Detected: {plates_detected}")
    print(f"📈 Average Confidence: {avg_confidence:.4f}")
    print(f"💾 Output saved to: {OUTPUT_FILE}")
    
else:
    print(f"❌ Path not found: {CUSTOM_TEST_DIR}")
    print("📝 Please update CUSTOM_TEST_DIR with your test set path")


🔍 RUNNING INFERENCE ON CUSTOM TEST SET
📂 Scanning custom test set: /Users/nguyenthientai/Documents/baseline_icpr_2026-main/pub_test
✅ Found 1000 tracks


Inferencing: 100%|██████████| 1000/1000 [00:26<00:00, 38.41it/s]


✅ Inference complete! Results saved to 'inference_results2.txt'
📊 Total predictions: 1000

📋 Sample Results (first 10):
   track_10005,FXB0728;0.9805
   track_10015,BAX0807;0.9863
   track_10016,MJQ4845;0.9926
   track_10035,BBZ8740;0.9990
   track_10039,ATC7553;0.9838
   track_10058,AWH9327;0.9896
   track_10073,BBX0749;0.9901
   track_10088,ARF3332;0.9310
   track_10089,BCY1440;0.9661
   track_10091,BBA6476;0.9883

📊 INFERENCE RESULTS SUMMARY
✅ Total Tracks: 1000
✅ Plates Detected: 1000
📈 Average Confidence: 0.9857
💾 Output saved to: inference_results2.txt


: 

: 

: 